# 1.1 — Ingestão de ordens de venda

- **Propósito:** Ingerir incrementalmente arquivos TXT do SAP e manter a tabela `raw_sales_order`.
- **Entrada:** Arquivos TXT delimitados por pipe
- **Saída:** `parts_hdbk_sandbox.dt_sales_orders.raw_sales_order`
- **Chave:** numero_ov + item · **Carga:** Incremental via Auto Loader

In [0]:
from delta.tables import DeltaTable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

from typing import Sequence

import uuid

In [0]:
# Constantes do Pipeline

# -----------------------------------------------------------------------------
# Layout do arquivo SAP
# -----------------------------------------------------------------------------

SALES_ORDER_LAYOUT = [
    "numero_ov",
    "data",
    "tipo_ov",
    "motivo_ov",
    "bloqueio_rem",
    "motivo_recusa",
    "org_vendas",
    "canal_dist",
    "setor_ativ",
    "centro",
    "emissor_da_ordem",
    "autor",
    "item",
    "material",
    "categoria_do_item",
    "item_superior",
    "quantidade",
    "um"
]

RAW_BUSINESS_COLUMNS = [
    "numero_ov", "data", "tipo_ov", "motivo_ov", "bloqueio_rem",
    "motivo_recusa", "org_vendas", "canal_dist", "setor_ativ", "centro",
    "emissor_da_ordem", "numero_pedido", "autor", "item", "material",
    "categoria_do_item", "item_superior", "quantidade", "um",
]
SOURCE_METADATA_COLUMNS = [
    "_source_file_name", "_source_file_path", "_source_file_modification_time",
]
AUDIT_COLUMNS = [
    "_ingested_at", "_last_updated_at", "_ingested_by", "_load_type", "_load_id",
]
RAW_COLUMNS = RAW_BUSINESS_COLUMNS + SOURCE_METADATA_COLUMNS + AUDIT_COLUMNS

# -----------------------------------------------------------------------------
# Chave de Negócio
# -----------------------------------------------------------------------------

BUSINESS_KEY = [
    "numero_ov",
    "item"
]

# -----------------------------------------------------------------------------
# Colunas para Padronização
# -----------------------------------------------------------------------------

DATE_COLUMNS = [
    "data"
]

NUMERIC_COLUMNS = [
    "quantidade"
]

# -----------------------------------------------------------------------------
# Trava de validação dos arquivos
# -----------------------------------------------------------------------------

# Organizações de vendas esperadas em cada período e o segmento correspondente
# (0200 = 2W, 0500 = 4W). Cada período precisa ter um arquivo por organização.
EXPECTED_ORGS = {
    "0200": "2W",
    "0500": "4W",
}

# Padrão do nome: salesorder_raw_<AAAA-MM>_<ORG>_<SEGMENTO>.txt
FILE_NAME_REGEX = r"^salesorder_raw_(\d{4}-\d{2})_(\d{4})_(\dW)\.txt$"


In [0]:
# Configuração do Catalog, Schema e Tabelas

# -----------------------------------------------------------------------------
# Catálogo
# -----------------------------------------------------------------------------

CATALOG = "parts_hdbk_sandbox"
SCHEMA = "dt_sales_orders"

# -----------------------------------------------------------------------------
# Tabelas
# -----------------------------------------------------------------------------

RAW_TABLE = f"{CATALOG}.{SCHEMA}.raw_sales_order"

# -----------------------------------------------------------------------------
# Caminhos
# -----------------------------------------------------------------------------

VOLUME_PATH = "/Volumes/parts_hdbk_sandbox/dt_sales_orders/sap_sales_order"

INPUT_PATH = f"{VOLUME_PATH}/incremental"

CHECKPOINT_PATH = f"{VOLUME_PATH}/_checkpoint/raw_sales_order"

SCHEMA_PATH = f"{VOLUME_PATH}/_schema/raw_sales_order"

# -----------------------------------------------------------------------------
# Auto Loader
# -----------------------------------------------------------------------------

AUTO_LOADER_FORMAT = "cloudFiles"
FILE_FORMAT = "text"
FILE_PATTERN = "*.txt"
INCLUDE_EXISTING_FILES = True
# Reprocessa arquivos reenviados com o mesmo nome (MERGE por chave evita duplicidade)
ALLOW_OVERWRITES = True

# -----------------------------------------------------------------------------
# Streaming
# -----------------------------------------------------------------------------

TRIGGER_AVAILABLE_NOW = True

# -----------------------------------------------------------------------------
# Modo de Carga
# -----------------------------------------------------------------------------
LOAD_MODE = "INCREMENTAL"
# Opções:
# "INCREMENTAL"
# "period_refresh"

LOAD_ID = str(uuid.uuid4())
CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]

SOURCE_METADATA_SCHEMA = {
    "_source_file_name": "STRING",
    "_source_file_path": "STRING",
    "_source_file_modification_time": "TIMESTAMP",
}

In [0]:
# Função de leitura incremental utilizando Auto Loader
# Configura schema inference e leitura de arquivos TXT delimitados por pipe

def read_sales_order() -> DataFrame:
    """
    Realiza a leitura incremental dos arquivos TXT exportados do SAP
    utilizando Auto Loader.

    Returns
    -------
    DataFrame
        Streaming DataFrame contendo uma linha por registro do arquivo.
    """

    reader = (
        spark.readStream
            .format(AUTO_LOADER_FORMAT)
            .option("cloudFiles.format", FILE_FORMAT)
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .option(
                "cloudFiles.includeExistingFiles",
                str(INCLUDE_EXISTING_FILES).lower()
            )
            .option(
                "cloudFiles.allowOverwrites",
                str(ALLOW_OVERWRITES).lower()
            )
            .option("pathGlobFilter", FILE_PATTERN)
    )

    return reader.load(INPUT_PATH)

In [0]:
# Função para iniciar o processamento em streaming
# Utiliza foreachBatch para processar micro-batches

def start_stream(df: DataFrame) -> None:
    """
    Inicia o processamento do pipeline utilizando Structured Streaming.
    """

    (
        df.writeStream
            .trigger(availableNow=TRIGGER_AVAILABLE_NOW)
            .option("checkpointLocation", CHECKPOINT_PATH)
            .foreachBatch(process_batch)
            .start()
            .awaitTermination()
    )

In [0]:
def process_batch(df: DataFrame, batch_id: int) -> None:
    """
    Processa um micro-batch recebido pelo Auto Loader.
    
    Etapas:
    1. Parse do arquivo SAP utilizando layout configurado
    2. Padronização de tipos de dados e formatos
    3. Trava de validação (aborta o batch inteiro se algum arquivo estiver errado)
    4. Deduplicação dentro do batch
    5. Persistência via MERGE na tabela Delta
    6. Log de métricas do batch processado
    """
    parsed_df = parse_sap_file(
        df,
        SALES_ORDER_LAYOUT
    )
    standardized_df = standardize_sales_order(
        parsed_df
    )

    # Nenhuma linha é gravada se qualquer arquivo do batch for reprovado
    batch_files = [r[0] for r in df.select("_source_file_name").distinct().collect()]
    assert_files_valid(standardized_df, f"batch {batch_id}", batch_files)

    deduplicated_df = deduplicate_batch(
        standardized_df
    )

    persist_sales_order(
        deduplicated_df
    )

    log_batch_metrics(batch_id)

In [0]:
# Função de parser para arquivos TXT exportados do SAP
# Divide a coluna value utilizando delimitador e aplica o layout configurado

def parse_sap_file(
    df: DataFrame,
    layout: Sequence[str]
) -> DataFrame:
    """
    Realiza o parse de arquivos TXT exportados do SAP.
    """

    expected_columns = len(layout)

    parsed_df = (
        df
        .filter(F.trim(F.col("value")) != "")
        .withColumn(
            "value",
            F.regexp_replace(F.col("value"), r"^\|", "")
        )
        .withColumn(
            "value",
            F.regexp_replace(F.col("value"), r"\|$", "")
        )
        .withColumn(
            "fields",
            F.split(F.col("value"), r"\|")
        )
        .filter(
            F.size(F.col("fields")) == expected_columns
        )
        .filter(
            ~F.col("fields")[0].startswith("Doc.")
        )
        .filter(
            F.trim(F.col("fields")[1]).rlike(r"^\d{2}\.\d{2}\.\d{4}$")
        )
    )

    return parsed_df.select(
    "_source_file_name",
    "_source_file_path",
    "_source_file_modification_time",
    *[
        F.trim(F.col("fields")[i]).alias(column)
        for i, column in enumerate(layout)
    ]
)

In [0]:
# Função de padronização de tipos de dados
# Converte tipos, formata datas e adiciona colunas de auditoria

def standardize_sales_order(df: DataFrame) -> DataFrame:
    """
    Padroniza os dados do relatório de Sales Order.
    """

    df = standardize_nulls(df)
    df = standardize_dates(df)
    df = standardize_numbers(df)

    df = (
        df
        .withColumn(
            "numero_pedido",
            F.lit(None).cast("string")
        )
    )

    df = add_metadata(df)

    return df.select(*RAW_COLUMNS)

def standardize_dates(df: DataFrame) -> DataFrame:
    """
    Converte colunas de data para o tipo Date.
    """

    for column in DATE_COLUMNS:
        df = df.withColumn(
            column,
            F.to_date(F.col(column), "dd.MM.yyyy")
        )

    return df


def standardize_numbers(df: DataFrame) -> DataFrame:
    """
    Converte colunas numéricas para Double.
    """

    for column in NUMERIC_COLUMNS:
        raw_value = F.trim(F.col(column))
        signed_value = F.when(
            raw_value.rlike(r".*-$"),
            F.concat(F.lit("-"), F.regexp_replace(raw_value, r"-$", "")),
        ).otherwise(raw_value)
        normalized_value = F.regexp_replace(
            F.regexp_replace(signed_value, r"\.", ""),
            ",",
            ".",
        )
        df = df.withColumn(column, normalized_value.cast("double"))

    return df


def standardize_nulls(df: DataFrame) -> DataFrame:
    """
    Converte strings vazias em NULL.
    """

    for column in df.schema.fieldNames():
        if not isinstance(df.schema[column].dataType, T.StringType):
            continue
        df = df.withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "",
                F.lit(None)
            ).otherwise(F.col(column))
        )

    return df

def add_metadata(df: DataFrame) -> DataFrame:
    """
    Adiciona metadados de ingestão.
    """

    return df.withColumns({
        "_ingested_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_last_updated_at": F.from_utc_timestamp(
            F.current_timestamp(), "America/Sao_Paulo"
        ),
        "_ingested_by": F.lit(CURRENT_USER),
        "_load_type": F.lit(LOAD_MODE),
        "_load_id": F.lit(LOAD_ID),
    })

In [0]:
# Função de deduplicação dentro do micro-batch
# Remove registros duplicados utilizando a chave de negócio definida

def deduplicate_batch(df: DataFrame) -> DataFrame:
    """
    Remove registros duplicados dentro do micro-batch.
    """

    return df.dropDuplicates(BUSINESS_KEY)

In [0]:
# Funções de persistência na tabela Delta
# Realiza MERGE incremental utilizando a chave de negócio

def ensure_raw_table_schema() -> None:
    """Migra o metadado legado e garante o schema de rastreabilidade da RAW."""

    if not spark.catalog.tableExists(RAW_TABLE):
        raise RuntimeError(f"Tabela RAW não encontrada: {RAW_TABLE}. Execute o notebook 00 primeiro.")

    existing_columns = set([field.name for field in spark.table(RAW_TABLE).schema.fields])
    missing_columns = [
        f"{name} {data_type}"
        for name, data_type in SOURCE_METADATA_SCHEMA.items()
        if name not in existing_columns
    ]
    if missing_columns:
        spark.sql(f"ALTER TABLE {RAW_TABLE} ADD COLUMNS ({', '.join(missing_columns)})")


def persist_sales_order(df: DataFrame) -> None:
    """
    Persiste os dados na tabela RAW conforme o modo de carga.
    """

    if LOAD_MODE == "PERIOD_REFRESH":
        period_refresh(df)
    else:
        incremental_load(df)

### INCREMENTAL LOAD

def incremental_load(df: DataFrame) -> None:
    """
    Realiza carga incremental utilizando MERGE.
    """

    target = DeltaTable.forName(spark, RAW_TABLE)

    merge_condition = " AND ".join(
        [f"t.{c} = s.{c}" for c in BUSINESS_KEY]
    )

    (
        target.alias("t")
        .merge(
            df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

### FULL REFRESH

def period_refresh(df: DataFrame) -> None:
    """
    Realiza carga completa da tabela.
    """

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RAW_TABLE)
    )

In [0]:
# Função de log de métricas do batch processado
# Registra contadores e estatísticas de processamento

def log_batch_metrics(batch_id: int) -> None:
    """
    Registra métricas básicas do micro-batch.
    """

    print("=" * 80)
    print(f"Batch ID........: {batch_id}")
    print("Persistência....: concluída")
    print(f"Tabela destino..: {RAW_TABLE}")
    print(f"Modo de carga...: {LOAD_MODE}")
    print("=" * 80)

In [0]:
# Trava de validação dos arquivos
# Se QUALQUER arquivo estiver errado, a carga inteira é abortada antes do MERGE
# (nenhuma linha é gravada e o checkpoint não avança).
#
# Regras por arquivo:
#   1. Nome no padrão salesorder_raw_<AAAA-MM>_<ORG>_<SEGMENTO>.txt
#   2. ORG conhecida e segmento coerente (0200 -> 2W, 0500 -> 4W)
#   3. Pelo menos uma linha válida após o parse
#   4. Todas as datas dentro do mês do nome do arquivo
#   5. org_vendas do conteúdo igual à ORG do nome
# Regras do conjunto:
#   6. No máximo um arquivo por (período, ORG)
#   7. Para cada período recebido, todas as ORGs de EXPECTED_ORGS precisam vir
#      juntas (ou já estar carregadas na tabela para aquele mês)

import os
import re
from datetime import date
from fnmatch import fnmatch


def collect_file_stats(df: DataFrame) -> list:
    """Calcula, por arquivo, as estatísticas usadas na validação."""

    return (
        df.groupBy("_source_file_name")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.when(F.col("data").isNull(), 1).otherwise(0)).alias("null_dates"),
            F.min(F.date_format("data", "yyyy-MM")).alias("min_period"),
            F.max(F.date_format("data", "yyyy-MM")).alias("max_period"),
            F.collect_set("org_vendas").alias("orgs"),
        )
        .collect()
    )


def period_loaded_in_table(period: str, org: str) -> bool:
    """Indica se a tabela RAW já possui dados da ORG no mês informado."""

    year, month = map(int, period.split("-"))
    start = date(year, month, 1)
    end = date(year + (month == 12), month % 12 + 1, 1)

    return (
        spark.table(RAW_TABLE)
        .filter(
            (F.col("org_vendas") == org)
            & (F.col("data") >= F.lit(start))
            & (F.col("data") < F.lit(end))
        )
        .limit(1)
        .count()
        > 0
    )


def validate_files(stats_rows: list, file_names: Sequence[str]) -> list:
    """Aplica as regras de validação e retorna a lista de erros encontrados."""

    errors = []
    stats = {row["_source_file_name"]: row for row in stats_rows}
    files_by_key = {}

    for name in sorted(set(file_names) | set(stats)):
        match = re.match(FILE_NAME_REGEX, name)
        if not match:
            errors.append(f"{name}: nome fora do padrão salesorder_raw_AAAA-MM_ORG_SEGMENTO.txt")
            continue

        period, org, segment = match.groups()
        if org not in EXPECTED_ORGS:
            errors.append(f"{name}: ORG {org} não esperada (válidas: {sorted(EXPECTED_ORGS)})")
            continue
        if EXPECTED_ORGS[org] != segment:
            errors.append(f"{name}: ORG {org} deve ter segmento {EXPECTED_ORGS[org]}, não {segment}")

        files_by_key.setdefault((period, org), []).append(name)

        row = stats.get(name)
        if row is None or row["rows"] == 0:
            errors.append(f"{name}: nenhuma linha válida após o parse")
            continue
        if row["null_dates"]:
            errors.append(f"{name}: {row['null_dates']} linhas sem data válida")
        if row["min_period"] != period or row["max_period"] != period:
            errors.append(
                f"{name}: datas de {row['min_period']} a {row['max_period']}, esperado somente {period}"
            )
        orgs = sorted(o for o in row["orgs"] if o is not None)
        if orgs != [org]:
            errors.append(f"{name}: org_vendas no conteúdo = {orgs}, esperado somente {org}")

    for (period, org), names in sorted(files_by_key.items()):
        if len(names) > 1:
            errors.append(f"período {period} ORG {org}: mais de um arquivo {names}")

    for period in sorted({period for period, _ in files_by_key}):
        for org, segment in EXPECTED_ORGS.items():
            if (period, org) not in files_by_key and not period_loaded_in_table(period, org):
                errors.append(
                    f"período {period}: falta o arquivo da ORG {org} ({segment}); "
                    "os arquivos do período precisam ser carregados juntos"
                )

    return errors


def assert_files_valid(
    df: DataFrame,
    context: str,
    file_names: Sequence[str] = None
) -> None:
    """Aborta a carga (RuntimeError) se qualquer arquivo for reprovado."""

    stats_rows = collect_file_stats(df)
    names = list(file_names) if file_names else [r["_source_file_name"] for r in stats_rows]
    errors = validate_files(stats_rows, names)

    if errors:
        raise RuntimeError(
            f"Carga abortada ({context}): nenhum dado foi gravado.\n - " + "\n - ".join(errors)
        )

    for row in sorted(stats_rows, key=lambda r: r["_source_file_name"]):
        print(
            f"[OK] {row['_source_file_name']}: {row['rows']} linhas, "
            f"período {row['min_period']}, org {sorted(row['orgs'])}"
        )


def list_pending_files() -> list:
    """Lista os arquivos da pasta de entrada que o Auto Loader ainda vai processar."""

    files = [f for f in dbutils.fs.ls(INPUT_PATH) if fnmatch(f.name, FILE_PATTERN)]

    processed = {}
    if os.path.exists(CHECKPOINT_PATH):
        for row in spark.sql(
            f"SELECT path, unix_millis(create_time) AS mtime FROM cloud_files_state('{CHECKPOINT_PATH}')"
        ).collect():
            processed[os.path.basename(row["path"])] = row["mtime"]

    return [
        f for f in files
        if f.name not in processed
        or (ALLOW_OVERWRITES and f.modificationTime > (processed[f.name] or 0))
    ]


def preflight_validate_files() -> None:
    """Valida os arquivos pendentes ANTES de iniciar o stream."""

    pending = list_pending_files()
    if not pending:
        print("Pré-validação: nenhum arquivo novo para carregar.")
        return

    print(f"Pré-validação de {len(pending)} arquivo(s): {[f.name for f in pending]}")

    df = (
        spark.read.text([f.path for f in pending])
        .withColumns({
            "_source_file_name": F.col("_metadata.file_name"),
            "_source_file_path": F.col("_metadata.file_path"),
            "_source_file_modification_time": F.col("_metadata.file_modification_time"),
        })
    )
    df_std = standardize_dates(
        standardize_nulls(parse_sap_file(df, SALES_ORDER_LAYOUT))
    )

    assert_files_valid(df_std, "pré-validação", [f.name for f in pending])

In [0]:
# =============================================================================
# Execução do Pipeline
# =============================================================================

# Trava: valida todos os arquivos pendentes antes de iniciar o stream.
# Se algum estiver errado, gera erro aqui e nada é carregado.
preflight_validate_files()

df_raw = read_sales_order()

# Capture metadata columns on the streaming DataFrame BEFORE entering foreachBatch
# (_metadata pseudo-columns are only available on streaming DataFrames, not on batch DataFrames)
df_with_metadata = df_raw.withColumns({
    "_source_file_name": F.col("_metadata.file_name"),
    "_source_file_path": F.col("_metadata.file_path"),
    "_source_file_modification_time": F.col("_metadata.file_modification_time"),
})

start_stream(df_with_metadata)

In [0]:
# Estatísticas da tabela após a carga (uma linha por carga e arquivo, mais recentes primeiro)

from pyspark.sql import functions as F

summary = (
    spark.table(RAW_TABLE)
         .groupBy("_load_id", "_ingested_by", "_ingested_at", "_source_file_name")
         .agg(
             F.count("*").alias("rows"),
             F.countDistinct("numero_ov").alias("orders"),
             F.min("data").alias("data_min"),
             F.max("data").alias("data_max")
         )
         .orderBy(F.col("_ingested_at").desc(), F.col("_source_file_name"))
         .withColumns({
             "_ingested_at": F.date_format(F.col("_ingested_at"), "yyyy/MM/dd HH:mm:ss"),
             "data_min": F.date_format(F.col("data_min"), "yyyy/MM/dd"),
             "data_max": F.date_format(F.col("data_max"), "yyyy/MM/dd"),
         })
)

display(summary)